# Titanic Survival Prediction with CatBoost

In this notebook, I use CatBoost to build a second model for the Titanic dataset. The goal is to compare this approach with the Logistic Regression baseline. 

In [1]:
import pandas as pd

train= pd.read_csv("../data/train.csv")

train.head()

,PassengerId,Survived,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked
0,1,0,3,"Braund, Mr. Owen Harris",male,22.0,1,0,A/5 21171,7.2500,NaN,S
1,2,1,1,"Cumings, Mrs. John Bradley (Florence Briggs Th...",female,38.0,1,0,PC 17599,71.2833,C85,C
2,3,1,3,"Heikkinen, Miss. Laina",female,26.0,0,0,STON/O2. 3101282,7.9250,NaN,S
3,4,1,1,"Futrelle, Mrs. Jacques Heath (Lily May Peel)",female,35.0,1,0,113803,53.1000,C123,S
4,5,0,3,"Allen, Mr. William Henry",male,35.0,0,0,373450,8.0500,NaN,S


In [2]:
from catboost import CatBoostClassifier

print("CatBoost imported successfully")

CatBoost imported successfully


# Exploring the Dataset

Before building the CatBoost model, I first load and inspect the dataset to understand its structure and identify any missing values that need to be addressed. 

In [3]:
train.isnull().sum()

PassengerId      0
Survived         0
Pclass           0
Name             0
Sex              0
Age            177
SibSp            0
Parch            0
Ticket           0
Fare             0
Cabin          687
Embarked         2
dtype: int64

# Cleaning the Data

The dataset contains missing values in several columns. I remove columns that are unlikely to be useful and handle the remaining missing values before training the model. 

In [4]:
train= train.drop(
    columns=["PassengerId", "Name", "Ticket", "Cabin"]
)

train.head()

,Survived,Pclass,Sex,Age,SibSp,Parch,Fare,Embarked
0,0,3,male,22.0,1,0,7.2500,S
1,1,1,female,38.0,1,0,71.2833,C
2,1,3,female,26.0,0,0,7.9250,S
3,1,1,female,35.0,1,0,53.1000,S
4,0,3,male,35.0,0,0,8.0500,S


In [5]:
train["Age"] = train["Age"].fillna(
    train["Age"].median()
)

train["Embarked"]= train["Embarked"].fillna(
    train["Embarked"].mode()[0]
)

train.isnull().sum()

Survived    0
Pclass      0
Sex         0
Age         0
SibSp       0
Parch       0
Fare        0
Embarked    0
dtype: int64

# Creating Featured and Target Variables

The target variable is Survived, which is what the model will learn to predict. The remaining columns are used as input features. 

In [6]:
X= train.drop("Survived", axis=1)
y= train["Survived"]

X.head()

,Pclass,Sex,Age,SibSp,Parch,Fare,Embarked
0,3,male,22.0,1,0,7.2500,S
1,1,female,38.0,1,0,71.2833,C
2,3,female,26.0,0,0,7.9250,S
3,1,female,35.0,1,0,53.1000,S
4,3,male,35.0,0,0,8.0500,S


# Splitting the Data

To evaluate the model fairly, I split the dataset into training and testing sets. The model is trained on one portion of the data and evaluated on data it has not seen before. 

In [7]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X, 
    y, 
    test_size= 0.2, 
    random_state=42
)

print(X_train.shape)
print(X_test.shape)

(712, 7)
(179, 7)


# Training the CatBoost Model

CatBoost can work directly with categorical variables, so I specify which columns are categorical and train the model using the training data. 

In [8]:
cat_features = ["Sex", "Embarked"]

model = CatBoostClassifier(
    iterations=500, 
    learning_rate=0.05, 
    depth=6,
    verbose=False
)

model.fit(
    X_train, 
    y_train, 
    cat_features= cat_features
)

CatBoostClassifier(depth=6, iterations=500, learning_rate=0.05, verbose=False)

# Evaluating Model Accuracy

After training the CatBoost model, I use it to make predictions on the test set and calculate the overall accuracy. 

In [9]:
from sklearn.metrics import accuracy_score

y_pred = model.predict(X_test)

accuracy= accuracy_score(y_test, y_pred)

print("Accuracy:", accuracy)

Accuracy: 0.8268156424581006


# Looking at Model Performance in More Detail

Accuracy provides an overall measure of performance, but the confusion matrix helps show where the model makes correct and incorrect predictions. 

In [10]:
from sklearn.metrics import confusion_matrix

confusion_matrix(y_test, y_pred)

array([[93, 12],
       [19, 55]])

# Classification Report

The classification report provides additional metrics including precision, recall, and F1-score for each class.

In [11]:
from sklearn.metrics import classification_report

print(classification_report(y_test, y_pred))

              precision    recall  f1-score   support

           0       0.83      0.89      0.86       105
           1       0.82      0.74      0.78        74

    accuracy                           0.83       179
   macro avg       0.83      0.81      0.82       179
weighted avg       0.83      0.83      0.83       179



# Conclusion

The CatBoost model achieved about 83% accuracy on the test set, slightly outperforming the Logistic Regression model from the previous notebook. One advantage of CatBoost is that it can work directly with categorical variables such as Sex and Embarked without requiring one-hot encoding. Overall, CatBoost produced strong results on the Titanic dataset and demonstrated why gradient boosted tree models are commonly used for tabular machine learning problems. 